In [8]:
"""
Sample fresh articles from the 182K corpus, excluding the 18K already used
"""

import pandas as pd

# ============================================================================
# LOAD DATA
# ============================================================================

print("="*60)
print("SAMPLING FRESH ARTICLES FROM 182K CORPUS")
print("="*60)

# Load the original 182K corpus
print("\nLoading 182K corpus...")
corpus_182k = pd.read_csv("/Users/FrederikkeB/Documents/GitHub/misleading-climate-claims-dk/data/raw/climate_misinfo_2023-2025.csv")  
print(f"Total corpus: {len(corpus_182k):,} articles")

# Load the 18K to get excluded keys
llm_18k = pd.read_parquet("/Users/FrederikkeB/Documents/GitHub/misleading-climate-claims-dk/data/processed/raw/llm_data_clean.parquet")  
excluded_keys = set(llm_18k['ArticleKey'])
print(f"Excluding {len(excluded_keys):,} articles from 18K sample")

# ============================================================================
# CREATE FRESH POOL
# ============================================================================

# Remove the 18K articles
df = corpus_182k[~corpus_182k['ArticleKey'].isin(excluded_keys)].reset_index(drop=True)
print(f"Fresh pool available: {len(df):,} articles")

SAMPLING FRESH ARTICLES FROM 182K CORPUS

Loading 182K corpus...
Total corpus: 182,584 articles
Excluding 17,976 articles from 18K sample
Fresh pool available: 164,608 articles


In [9]:
# ============================================================================
# PREPARE STRATIFICATION VARIABLE
# ============================================================================

# set size of subset
TOTAL_SAMPLE = 10000
RANDOM_STATE = 55

# compute year distribution
year_props = df["Year"].value_counts(normalize=True)

# compute target size per year
year_targets = (year_props * TOTAL_SAMPLE).round().astype(int)

sampled_list = []

In [10]:
# sample data
for year, year_n in year_targets.items():
    
    df_year = df[df["Year"] == year]
    
    # proportion per media within that year
    media_props = df_year["Media"].value_counts(normalize=True)
    media_targets = (media_props * year_n).round().astype(int)
    
    for media, media_n in media_targets.items():
        df_media = df_year[df_year["Media"] == media]
        
        # avoid sampling more than what is possible 
        media_n = min(media_n, len(df_media))
        
        sampled_media = df_media.sample(
            n=media_n,
            random_state=RANDOM_STATE
        )
        
        sampled_list.append(sampled_media)

final_sample = pd.concat(sampled_list).reset_index(drop=True)

print("Final sample size:", final_sample.shape)
print(final_sample["Year"].value_counts(normalize=True))

Final sample size: (9375, 21)
Year
2023    0.416213
2024    0.323307
2025    0.260480
Name: proportion, dtype: float64


In [11]:
final_sample.to_excel("fresh_sample_9K.xlsx")

In [13]:
"""
Sample fresh articles for Active Learning Round 2
Excludes: 18K LLM sample + Round 1 9K sample
"""
import pandas as pd

# ============================================================================
# LOAD DATA
# ============================================================================
print("="*60)
print("ACTIVE LEARNING ROUND 2: SAMPLING FRESH ARTICLES")
print("="*60)

# Load the original 182K corpus
print("\nLoading 182K corpus...")
corpus_182k = pd.read_csv("/Users/FrederikkeB/Documents/GitHub/misleading-climate-claims-dk/data/raw/climate_misinfo_2023-2025.csv")  
print(f"Total corpus: {len(corpus_182k):,} articles")

# Load the 18K to get excluded keys
print("\nLoading exclusions...")
llm_18k = pd.read_parquet("/Users/FrederikkeB/Documents/GitHub/misleading-climate-claims-dk/data/interim/sample_llm_18k.parquet")  
excluded_keys = set(llm_18k['ArticleKey'])
print(f"  - Excluding {len(excluded_keys):,} articles from 18K sample")

# Load Round 1 9K and add to exclusions
round1_9k = pd.read_excel("/Users/FrederikkeB/Documents/GitHub/misleading-climate-claims-dk/data/interim/sample_active_learning_9k.xlsx")  
excluded_keys.update(round1_9k['ArticleKey'])
print(f"  - Excluding {len(round1_9k):,} articles from Round 1 9K sample")
print(f"  - Total unique excluded: {len(excluded_keys):,} articles")

# ============================================================================
# CREATE FRESH POOL
# ============================================================================
# Remove the excluded articles
df = corpus_182k[~corpus_182k['ArticleKey'].isin(excluded_keys)].reset_index(drop=True)
print(f"\nFresh pool available: {len(df):,} articles")

ACTIVE LEARNING ROUND 2: SAMPLING FRESH ARTICLES

Loading 182K corpus...
Total corpus: 182,584 articles

Loading exclusions...
  - Excluding 18,008 articles from 18K sample
  - Excluding 9,375 articles from Round 1 9K sample
  - Total unique excluded: 27,381 articles

Fresh pool available: 155,203 articles


In [15]:
# ============================================================================
# PREPARE STRATIFICATION VARIABLE
# ============================================================================

# set size of subset
TOTAL_SAMPLE = 20000
RANDOM_STATE = 55

# compute year distribution
year_props = df["Year"].value_counts(normalize=True)

# compute target size per year
year_targets = (year_props * TOTAL_SAMPLE).round().astype(int)

sampled_list = []

In [16]:
# sample data
for year, year_n in year_targets.items():
    
    df_year = df[df["Year"] == year]
    
    # proportion per media within that year
    media_props = df_year["Media"].value_counts(normalize=True)
    media_targets = (media_props * year_n).round().astype(int)
    
    for media, media_n in media_targets.items():
        df_media = df_year[df_year["Media"] == media]
        
        # avoid sampling more than what is possible 
        media_n = min(media_n, len(df_media))
        
        sampled_media = df_media.sample(
            n=media_n,
            random_state=RANDOM_STATE
        )
        
        sampled_list.append(sampled_media)

final_sample = pd.concat(sampled_list).reset_index(drop=True)

print("Final sample size:", final_sample.shape)
print(final_sample["Year"].value_counts(normalize=True))

Final sample size: (19724, 21)
Year
2023    0.409957
2024    0.328584
2025    0.261458
Name: proportion, dtype: float64


In [17]:
final_sample.to_excel("/Users/FrederikkeB/Documents/GitHub/misleading-climate-claims-dk/data/interim/sample_active_learning_20k.xlsx")